# S0: Temporal Persistence Profiling

**Goal**: Measure oracle δ = ‖h(t) - h(t-1)‖ / ‖h(t-1)‖ (ground truth channel variation)

**Key questions**:
1. How much does the channel change between consecutive slots? (go/no-go)
2. Static δ ≈ 0? (simulator noise floor sanity check)  
3. NMSE if we reuse previous estimate?

Run cells sequentially. Data loads once and stays in memory.

In [1]:
import sys; sys.path.insert(0, "../../..")  # project root
from src.experiments.S0_persistence.core import load_data, run_all_bs, check_go_nogo, save_results, load_results
import numpy as np
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-whitegrid")

PRESET = "munich_elaa_m_1k_15g"  # has h5 data ready
MAX_SNAPSHOTS = 20000

In [ ]:
# Load data (slow — stays in memory for re-analysis)
# Or load cached results if available
GPU = "0"             # GPU(s) to use, e.g. "0", "0,1,2,3"
UE_PER_SPEED = 1      # 1 UE per speed category = 3 UEs total (None = all)
SNAP_LIMIT = 10000    # snapshots per UE (None = all)

result = load_results(PRESET)
if result is None:
    print("No cached results, computing...")
    data = load_data(PRESET, max_snapshots=MAX_SNAPSHOTS)
    result = run_all_bs(data, PRESET, gpu=GPU, max_snapshots=SNAP_LIMIT, ue_per_speed=UE_PER_SPEED)
    save_results(result)
    print("Saved results. Data stays in memory.")
else:
    print(f"Loaded cached results: {result['summary']['n_ues']} UEs")

per_ue = result["per_ue"]
summary = result["summary"]
print(f"\nSummary: median δ={summary['median_delta']:.4f}, "
      f"static δ={summary['static_median_delta']:.4f}, "
      f"NMSE(reuse)={summary['median_nmse_reuse_db']:.1f}dB")

In [ ]:
# Plot 1: Per-UE scatter — speed vs median δ
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

speeds = np.array([u["speed"] for u in per_ue])
deltas = np.array([u["median_delta"] for u in per_ue])
nmses = np.array([u["nmse_reuse_db"] for u in per_ue])
dists = np.array([u["dist"] for u in per_ue])

# 1a: Speed vs δ
ax = axes[0]
scatter = ax.scatter(speeds + np.random.normal(0, 0.1, len(speeds)), deltas, 
                     c=dists, cmap="viridis", alpha=0.7, edgecolors="k", linewidths=0.5)
ax.axhline(summary["static_median_delta"], color="red", ls="--", alpha=0.5, label=f"static noise floor ({summary['static_median_delta']:.4f})")
ax.set_xlabel("Speed (m/s)")
ax.set_ylabel("Median δ_oracle")
ax.set_title("Speed → Channel Variation")
ax.legend(fontsize=8)
plt.colorbar(scatter, ax=ax, label="Distance (m)")

# 1b: Bar by speed category
ax = axes[1]
cats = {"static": 0.0, "ped (1m/s)": 1.0, "veh (8.3m/s)": 8.3}
for i, (label, spd) in enumerate(cats.items()):
    mask = np.abs(speeds - spd) < 0.1
    if mask.any():
        med = np.median(deltas[mask])
        ax.bar(i, med, color=["blue", "green", "red"][i], alpha=0.7, label=f"{label}: {med:.4f}")
ax.axhline(summary["static_median_delta"], color="red", ls="--", alpha=0.3)
ax.set_xticks(range(len(cats)))
ax.set_xticklabels(cats.keys())
ax.set_ylabel("Median δ_oracle")
ax.set_title("δ by Mobility")
ax.legend(fontsize=8)

# 1c: NMSE(reuse) by speed
ax = axes[2]
for i, (label, spd) in enumerate(cats.items()):
    mask = np.abs(speeds - spd) < 0.1
    valid = nmses[mask]
    valid = valid[~np.isnan(valid)]
    if len(valid) > 0:
        med = np.median(valid)
        ax.bar(i, med, color=["blue", "green", "red"][i], alpha=0.7, label=f"{label}: {med:.1f}dB")
ax.axhline(-10, color="gray", ls=":", alpha=0.5, label="-10dB threshold")
ax.set_xticks(range(len(cats)))
ax.set_xticklabels(cats.keys())
ax.set_ylabel("NMSE(reuse) [dB]")
ax.set_title("Quality Loss from Reuse")
ax.legend(fontsize=8)

fig.suptitle(f"S0: {PRESET} ({summary['n_ues']} UEs, {MAX_SNAPSHOTS} snaps)", fontsize=12)
plt.tight_layout()
plt.show()

## GO / NO-GO

In [ ]:
check_go_nogo([result])